In [ ]:
#!pip install ultralytics opencv-python pillow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import shutil
import random
from sklearn.model_selection import train_test_split
from PIL import Image
import yaml
import numpy as np
import cv2
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader, random_split
import timm
from sklearn.metrics import accuracy_score, f1_score
from itertools import product
from tqdm import tqdm
from sklearn.model_selection import StratifiedShuffleSplit

In [ ]:
path = os.getcwd()
print(path)

#os.chdir(path)
#file_log = open(path + "/mensagem_final_classificar_V2.txt", "a")

/content


In [ ]:
# === Configurações Gerais ===
NUM_CLASSES = 15
INPUT_SIZE = 224  # conforme modelo Small treinado em 224×224 (OBS: Entrada dobra de tamanho, logo 448 é o tamanho final após concatenação (só para referência))
BATCH_SIZE = 32 # Padrão: 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATASET_FOLHA = "/content/drive/MyDrive/TCC/Datasets/Imagens Folhas/Especies"
DATASET_CASCA = "/content/drive/MyDrive/TCC/Datasets/Imagens tronco/EspeciesCascas"

CKPT_DIR = "/content/drive/MyDrive/TCC/Datasets/checkpointsHybridInputMobileNet"
#FINAL_PATH = "/content/drive/MyDrive/TCC/Datasets/main_weights/hybrid_input_mobilenet_best.pt"
FINAL_DIR = "/content/drive/MyDrive/TCC/Datasets/main_weights"

FUSION_DATASET_TYPE = "cartesiano" # "cartesiano" ou "unitario"


In [ ]:
#  ===Dataset customizado ===
class ImageFolderDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        # Assumindo estrutura: root_dir/class_x/imagename.jpg
        self.samples = []
        classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls_name:i for i,cls_name in enumerate(classes)}
        for cls_name in classes:
            cls_path = os.path.join(root_dir, cls_name)
            if not os.path.isdir(cls_path):
                continue
            for fname in os.listdir(cls_path):
                if fname.lower().endswith(('.png','.jpg','.jpeg','.bmp')):
                    self.samples.append((os.path.join(cls_path, fname), self.class_to_idx[cls_name]))
        # embaralhar?
        np.random.seed(42)
        np.random.shuffle(self.samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        # leitura via OpenCV
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        # redimensionar
        img = cv2.resize(img, (INPUT_SIZE, INPUT_SIZE))
        # converter para float32 e normalizar [0,1]
        img = img.astype(np.float32) / 255.0
        # talvez normalização adicional conforme modelo (media/std)
        # usando valores padrão ImageNet
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        img = (img - mean) / std
        # mudar de H×W×C para C×H×W
        img = np.transpose(img, (2,0,1))
        img_tensor = torch.from_numpy(img)
        label_tensor = torch.tensor(label, dtype=torch.long)
        return img_tensor, label_tensor

In [ ]:
class CartesianFusionDataset(Dataset):
    def __init__(self, ds_folha, ds_casca):
        self.samples = []

        # Extrair samples considerando Subset
        def get_samples(ds):
            if isinstance(ds, torch.utils.data.Subset):
                # Pegar apenas os índices do subset
                base_samples = ds.dataset.samples
                return [base_samples[i] for i in ds.indices]
            else:
                return ds.samples

        folha_samples = get_samples(ds_folha)
        casca_samples = get_samples(ds_casca)

        # Agrupar por classe
        folhas_por_classe = {}
        cascas_por_classe = {}

        for img, label in folha_samples:
            if label not in folhas_por_classe:
                folhas_por_classe[label] = []
            folhas_por_classe[label].append(img)

        for img, label in casca_samples:
            if label not in cascas_por_classe:
                cascas_por_classe[label] = []
            cascas_por_classe[label].append(img)

        if FUSION_DATASET_TYPE == "cartesiano":
          # Produto cartesiano por classe
          for label in folhas_por_classe.keys():
              if label not in cascas_por_classe:
                  continue
              for f_img in folhas_por_classe[label]:
                  for c_img in cascas_por_classe[label]:
                      self.samples.append((f_img, c_img, label))
        elif FUSION_DATASET_TYPE == "unitario":
          # Combinação pareada por classe (com reinício circular das cascas)
          for label in folhas_por_classe.keys():
              if label not in cascas_por_classe:
                  continue
              folhas = folhas_por_classe[label]
              cascas = cascas_por_classe[label]
              for i, f_img in enumerate(folhas):
                  c_img = cascas[i % len(cascas)]
                  self.samples.append((f_img, c_img, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        folha_path, casca_path, label = self.samples[idx]

        # Preprocessar imagens
        img_f = self.preprocess(folha_path)
        img_c = self.preprocess(casca_path)

        return img_f, img_c, torch.tensor(label, dtype=torch.long)

    def preprocess(self, img_path):
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (INPUT_SIZE, INPUT_SIZE))
        img = img.astype(np.float32) / 255.0

        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        img = (img - mean) / std

        img = np.transpose(img, (2, 0, 1))
        return torch.from_numpy(img)

In [ ]:
def stratified_split(dataset, test_split=0.1, valid_split=0.2, seed=42):
    """
    Divide o dataset em treino, validação e teste mantendo a proporção das classes.

    Args:
        dataset: Dataset com atributo 'samples' [(path, label), ...]
        test_split: Proporção do teste em relação ao total (padrão: 0.1 = 10%)
        valid_split: Proporção da validação em relação ao (treino+validação) (padrão: 0.2 = 20%)
        seed: Seed para reprodutibilidade

    Returns:
        train_ds, valid_ds, test_ds: Subsets estratificados

    Nota: Com os valores padrão, a divisão real é:
        - Treino: 72% do total
        - Validação: 18% do total
        - Teste: 10% do total
    """
    # Extrair os rótulos
    labels = [label for _, label in dataset.samples]

    # Primeiro, separar Teste
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=test_split, random_state=seed)
    train_valid_idx, test_idx = next(sss1.split(np.zeros(len(labels)), labels))

    # Agora, separar Validação dentro do conjunto de treino+validação
    labels_train_valid = np.array(labels)[train_valid_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=valid_split, random_state=seed)
    train_idx, valid_idx = next(sss2.split(np.zeros(len(labels_train_valid)), labels_train_valid))

    # Reindexar para o dataset original
    train_idx = np.array(train_valid_idx)[train_idx]
    valid_idx = np.array(train_valid_idx)[valid_idx]

    # Criar Subsets
    train_ds = torch.utils.data.Subset(dataset, train_idx)
    valid_ds = torch.utils.data.Subset(dataset, valid_idx)
    test_ds  = torch.utils.data.Subset(dataset, test_idx)

    return train_ds, valid_ds, test_ds

In [ ]:
# === Função para treinar por uma época ===
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    for img_f, img_c, labels in tqdm(loader, desc="Train"):
        img_f, img_c, labels = img_f.to(DEVICE), img_c.to(DEVICE), labels.to(DEVICE)
        #imgs = cv2.vconcat([img_f, img_c])
        imgs = torch.cat([img_f, img_c], dim=2)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
    epoch_loss = running_loss / len(loader.dataset)
    return epoch_loss

# === Função para avaliar no conjunto de validação/teste ===
def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for img_f, img_c, labels in tqdm(loader, desc="Eval"):
            img_f, img_c, labels = img_f.to(DEVICE), img_c.to(DEVICE), labels.to(DEVICE)
            imgs = torch.cat([img_f, img_c], dim=2)
            labels = labels.to(DEVICE)
            outputs = model(imgs)
            _, preds = torch.max(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='weighted')
    return acc, f1

# === Função para montar o modelo adaptado para 15 classes ===
def create_model(num_classes=NUM_CLASSES, pretrained=True):
    model = timm.create_model(
      'mobilenetv4_conv_small.e1200_r224_in1k',
      pretrained=pretrained#,
      #img_size=(448, 224)  # altura dobrada
    )

    # Descongela tudo
    for param in model.parameters():
        param.requires_grad = True

    # Congela apenas o final
    for name, param in model.named_parameters():
        # Congela os primeiros blocos/stages da MobileNet
        # Congela 3 stages = 60% (mais conservador)
        if any(layer in name for layer in ['conv_stem', 'stages.0', 'stages.1', 'stages.2']):
            param.requires_grad = False

    model.reset_classifier(num_classes=num_classes)
    return model.to(DEVICE)

In [ ]:
# === Função principal para rodar Grid Search sobre epochs e learning rate ===
def run_grid_search(dataset_leaf, dataset_bark, lr_list, epoch_list, valid_split=0.2, test_split=0.1, patience=5):
    # carregar dataset completo

    ds_f = ImageFolderDataset(dataset_leaf)
    total_len = len(ds_f)
    test_len  = int(total_len * test_split)
    valid_len = int((total_len - test_len) * valid_split)
    train_len = total_len - test_len - valid_len
    train_f, valid_f, test_f = stratified_split(ds_f, test_split=test_split, valid_split=valid_split, seed=42)
    print(f"Dataset de Folhas:\nClasses detectadas ({len(ds_f.class_to_idx.keys())}): {ds_f.class_to_idx.keys()}")
    print(f"\nDataset total: {total_len} imagens")
    print(f"Treino: {train_len} | Validação: {valid_len} | Teste: {test_len}")

    ds_c = ImageFolderDataset(dataset_bark)
    total_len = len(ds_c)
    test_len  = int(total_len * test_split)
    valid_len = int((total_len - test_len) * valid_split)
    train_len = total_len - test_len - valid_len
    train_c, valid_c, test_c = stratified_split(ds_c, test_split=test_split, valid_split=valid_split, seed=42)
    print(f"\nDataset de Cascas:\nClasses detectadas ({len(ds_c.class_to_idx.keys())}): {ds_c.class_to_idx.keys()}")
    print(f"\nDataset total: {total_len} imagens")
    print(f"Treino: {train_len} | Validação: {valid_len} | Teste: {test_len}")

    #Criar produto cartesiano DEPOIS
    train_fusion = CartesianFusionDataset(train_f, train_c)
    valid_fusion = CartesianFusionDataset(valid_f, valid_c)
    test_fusion = CartesianFusionDataset(test_f, test_c)

    # Extra. Exibir dados do Dataset ---------------------------------------- DEBUG ------------------------------------------------------------------------------------------
    print(f"Total: {len(train_fusion)+len(valid_fusion)+len(test_fusion)} | Treino: {len(train_fusion)} | Validação: {len(valid_fusion)} | Teste: {len(test_fusion)}")

    train_loader = DataLoader(train_fusion, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
    valid_loader = DataLoader(valid_fusion, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
    test_loader  = DataLoader(test_fusion,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

    best_config = None
    best_f1 = 0.0 # Variável usada para o teste final para saber qual é a melhor configuração
    best_val_f1 = 0.0 # Variável usada para o Early Stopping
    early_stop = False
    patience_counter = 0

    for lr, epochs in product(lr_list, epoch_list):
        print(f"\n==== Training with lr={lr}, epochs={epochs} ====")
        best_val_f1 = 0.0 # Reseta sempre que começa uma nova configuração de hiperparâmetros
        model = create_model(num_classes=NUM_CLASSES, pretrained=True)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        # === Novo: tentar carregar último checkpoint existente ===
        #ckpt_dir = "checkpointsMobileNet" # Para salvar localmente, não no Drive
        ckpt_dir = CKPT_DIR
        os.makedirs(ckpt_dir, exist_ok=True)

        # Procura checkpoints com o padrão "mobilenetv4_epochX_eY_lrZ.pt" (X = época atual, Y = número de épocas máximas, Z = Learning Rate)
        ckpt_files = [f for f in os.listdir(ckpt_dir) if f"e{epochs:.0e}_lr{lr:.0e}" in f]
        last_epoch = 0
        if ckpt_files:
            # Ordena checkpoints por número da época
            ckpt_files.sort(key=lambda x: int(x.split("_epoch")[1].split("_")[0]))
            last_ckpt = os.path.join(ckpt_dir, ckpt_files[-1])
            print(f"Checkpoint detectado: {last_ckpt} — retomando treinamento...")
            checkpoint = torch.load(last_ckpt, map_location=DEVICE)
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            last_epoch = checkpoint['epoch']
            # Testa se já chegou no final do treinamento
            if last_epoch == epochs:
                print("Treinamento já foi finalizado!")
                loss, acc_val, f1_val = checkpoint['loss'], checkpoint['val_acc'], checkpoint['val_f1']
                print(f"Epoch {last_epoch}/{epochs} - loss: {loss:.7f}, val_acc: {acc_val:.4f}, val_f1: {f1_val:.4f}")
            else:
                print(f"Retomando a partir da época {last_epoch+1}/{epochs} (lr={lr})")
        else:
            print("Nenhum checkpoint anterior encontrado, começando do zero.")

        for epoch in range(last_epoch, epochs): # OBS: Se já tiver terminado, nem entra nesse loop
            loss = train_one_epoch(model, train_loader, criterion, optimizer)
            acc_val, f1_val = evaluate(model, valid_loader)
            print(f"Epoch {epoch+1}/{epochs} - loss: {loss:.7f}, val_acc: {acc_val:.4f}, val_f1: {f1_val:.4f}")

          # === Early Stopping por val_f1 ===
            if f1_val > best_val_f1:
                best_val_f1 = f1_val
                patience_counter = 0
                best_model_state = model.state_dict()
                best_epoch = epoch + 1
            else:
                patience_counter += 1
                print(f"Contador do Early stopping: {patience_counter}/{patience} - Melhor F1-Score anterior: {best_val_f1}")
                if patience_counter >= patience:
                    print(f"Parando antecipadamente na época {epoch+1}. Melhor val_f1: {best_val_f1:.4f}")
                    early_stop = True

        # === salvar checkpoint por época ===
            #ckpt_dir = "checkpointsMobileNet" # Para salvar localmente, não no Drive
            ckpt_dir = CKPT_DIR
            os.makedirs(ckpt_dir, exist_ok=True)
            if early_stop: # Salva a época atual como se fosse a última caso
              ckpt_path = os.path.join(ckpt_dir, f"mobilenetv4_epoch{epochs}_e{epochs:.0e}_lr{lr:.0e}.pt")
              current_epoch = epochs
              #saved_model_state = best_model_state # Se for parar, salva usando o melhor modelo que tinha achado
            else:
              current_epoch = epoch + 1
              ckpt_path = os.path.join(ckpt_dir, f"mobilenetv4_epoch{current_epoch}_e{epochs:.0e}_lr{lr:.0e}.pt")
              #saved_model_state = model.state_dict()
            torch.save({
                'epoch': current_epoch,
                'model_state_dict': model.state_dict(), #saved_model_state,
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': acc_val,
                'val_f1': f1_val,
                'loss': loss,
                'lr': lr
            }, ckpt_path)
            print(f"Checkpoint salvo: {ckpt_path}")
            ckpt_path = os.path.join(ckpt_dir, f"mobilenetv4_epoch{epoch}_e{epochs:.0e}_lr{lr:.0e}.pt")
            if os.path.exists(ckpt_path):
              os.remove(ckpt_path) # Apaga o checkpoint anterior
              print(f"Checkpoint deletado: {ckpt_path}")
            if early_stop:
              break # Early Stopping se o F1 Score não melhorar em 5 iterações

        # avaliar no conjunto de validação final
        early_stop = False
        patience_counter = 0 # Resetando a parte do Early Stopping
        if f1_val > best_f1:
            best_f1 = f1_val
            best_config = {'lr': lr, 'epochs': epochs, 'model_state': model.state_dict()}
            print(f"*** New best config: lr={lr}, epochs={epochs}, val_f1={f1_val:.4f} ***")

    # Depois de escolher melhor configuração, reavaliar no test set
    print(f"\nBest config found: {best_config['lr']=}, {best_config['epochs']=}, best_val_f1={best_f1:.4f}")
    # carregar o melhor modelo
    best_model = create_model(num_classes=NUM_CLASSES, pretrained=False)
    best_model.load_state_dict(best_config['model_state'])
    best_model.to(DEVICE)

    acc_test, f1_test = evaluate(best_model, test_loader)
    print(f"\nTest set - acc: {acc_test:.6f}, f1: {f1_test:.6f}")

    # === Novo: salvar pesos finais do melhor modelo ===
    #final_dir = "main_weights" # Para salvar localmente, não no Drive
    final_dir = FINAL_DIR #"/content/drive/MyDrive/TCC/Datasets/main_weights"
    os.makedirs(final_dir, exist_ok=True)
    final_path = os.path.join(final_dir, "hybrid_input_mobilenet_best.pt")
    torch.save(best_model.state_dict(), final_path)
    print(f"Pesos finais salvos em: {final_path}")

    return best_config, (acc_test, f1_test)

In [ ]:
# === Função Main ===
if __name__ == "__main__":
    try:
        print("\n--------------- Treinamento do modelo MobileNetV4 ---------------\nInício...")
        # Defina aqui o caminho para o diretório de imagens
        dataset_leaf = DATASET_FOLHA
        dataset_bark = DATASET_CASCA
        # Defina lista de valores para hyperparâmetros
        lr_list    = [5e-4]
        epoch_list = [10]
        # Rodar grid
        best_config, test_metrics = run_grid_search(dataset_leaf, dataset_bark, lr_list, epoch_list, valid_split=0.2, test_split=0.1, patience=2)
        print("Finished. Test metrics:", test_metrics)
        print("...Fim\n")
    except KeyboardInterrupt:
        print("Programa encerrado via terminal...")


--------------- Treinamento do modelo MobileNetV4 ---------------
Início...
Dataset de Folhas:
Classes detectadas (15): dict_keys(['Abacateiro', 'Araca', 'Brinco de Indio', 'Cajueiro', 'Carvalho', 'Caterete', 'Cerejeira', 'Coite', 'Fruta do conde', 'Grevilha', 'Jambolao', 'Laranja Champanhe', 'Louro Pardo', 'Pau Brasil', 'Peroba Rosa'])

Dataset total: 704 imagens
Treino: 508 | Validação: 126 | Teste: 70
Dataset de Cascas:
Classes detectadas (15): dict_keys(['Abacateiro', 'Araca', 'Brinco de Indio', 'Cajueiro', 'Carvalho', 'Caterete', 'Cerejeira', 'Coite', 'Fruta do conde', 'Grevilha', 'Jambolao', 'Laranja Champanhe', 'Louro Pardo', 'Pau Brasil', 'Peroba Rosa'])

Dataset total: 489 imagens
Treino: 353 | Validação: 88 | Teste: 48
Total: 12731 | Treino: 11765 | Validação: 738 | Teste: 228

==== Training with lr=0.0005, epochs=10 ====


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/15.2M [00:00<?, ?B/s]

Checkpoint detectado: /content/drive/MyDrive/TCC/Datasets/checkpointsHybridInput/mobilenetv4_epoch9_e1e+01_lr5e-04.pt — retomando treinamento...
Retomando a partir da época 10/10 (lr=0.0005)


Train:   0%|          | 0/368 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Eval: 100%|██████████| 24/24 [03:36<00:00,  9.02s/it]


Epoch 10/10 - loss: 0.0000491, val_acc: 0.9512, val_f1: 0.9519
Checkpoint salvo: /content/drive/MyDrive/TCC/Datasets/checkpointsHybridInput/mobilenetv4_epoch10_e1e+01_lr5e-04.pt
Checkpoint deletado: /content/drive/MyDrive/TCC/Datasets/checkpointsHybridInput/mobilenetv4_epoch9_e1e+01_lr5e-04.pt
*** New best config: lr=0.0005, epochs=10, val_f1=0.9519 ***

Best config found: best_config['lr']=0.0005, best_config['epochs']=10, best_val_f1=0.9519


Eval:   0%|          | 0/8 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Eval: 100%|██████████| 8/8 [01:12<00:00,  9.09s/it]



Test set - acc: 0.973684, f1: 0.973184
Pesos finais salvos em: /content/drive/MyDrive/TCC/Datasets/main_weights/hybrid_input_mobilenet_best.pt
Finished. Test metrics: (0.9736842105263158, 0.9731838516508351)
...Fim

